# Bielik — generacja przykładów rzadkich klas (augmentacja LLM)

**Wymaga:** GPU **T4×2** (4-bit/bitsandbytes), Internet ON, dataset `pl-emotion-processed`.

In [ ]:
!pip install -q -U "transformers>=4.44" "bitsandbytes>=0.43" accelerate 2>/dev/null
import torch, transformers; print(transformers.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
import os, glob, re, warnings
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
warnings.filterwarnings("ignore")
RANDOM_STATE=42; torch.manual_seed(RANDOM_STATE)
EMOTIONS=["radość","smutek","zaufanie","wstręt","strach","gniew","przeczuwanie","zdziwienie"]
RARE=["strach","zaufanie","smutek"]
MODEL_NAME="speakleash/Bielik-11B-v2.3-Instruct"
GEN_PER_CLASS=400; BATCH=16; MAX_NEW=64; OUT="/kaggle/working"

In [ ]:
def find_csv(n):
    h=glob.glob(f"/kaggle/input/**/{n}",recursive=True)
    if not h: raise FileNotFoundError(n)
    return h[0]
tw=pd.read_csv(find_csv("twitteremo_train.csv")); tw["tekst"]=tw["tekst"].fillna("")
# kilka prawdziwych przykładów per rzadka emocja (few-shot)
def examples_for(emo, k=5):
    sub=tw[(tw[emo]==1)]
    sub=sub[sub["tekst"].str.len().between(20,160)]
    return sub["tekst"].sample(min(k,len(sub)),random_state=RANDOM_STATE).tolist()
FEWSHOT={e:examples_for(e) for e in RARE}
print({e:len(v) for e,v in FEWSHOT.items()})

In [ ]:
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
tok=AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None: tok.pad_token=tok.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_NAME,quantization_config=bnb,device_map="auto")
model.eval()

In [ ]:
def build_prompt(emo):
    ex="\n".join(f"- {t}" for t in FEWSHOT[emo])
    msg=(f"Napisz jeden krótki, naturalny tweet po polsku, który wyraża emocję: {emo}. "
         f"Tweet ma brzmieć jak prawdziwy wpis z Twittera (potoczny, do ~200 znaków), bez hashtagów i bez cudzysłowów. "
         f"Przykłady tweetów wyrażających {emo}:\n{ex}\n\nNapisz tylko jeden nowy tweet:")
    return tok.apply_chat_template([{"role":"user","content":msg}],tokenize=False,add_generation_prompt=True)

@torch.no_grad()
def generate(emo,n):
    outs=[]
    while len(outs)<n:
        prompts=[build_prompt(emo) for _ in range(min(BATCH,n-len(outs)))]
        enc=tok(prompts,return_tensors="pt",padding=True,truncation=True,max_length=512).to(model.device)
        gen=model.generate(**enc,max_new_tokens=MAX_NEW,do_sample=True,temperature=0.9,top_p=0.95,pad_token_id=tok.pad_token_id)
        for i in range(len(prompts)):
            txt=tok.decode(gen[i][enc["input_ids"].shape[1]:],skip_special_tokens=True).strip()
            txt=txt.split("\n")[0].strip().strip('"').strip()
            if 10<len(txt)<=240: outs.append(txt)
    return outs[:n]

In [ ]:
rows=[]
for emo in RARE:
    print(f"generuję {GEN_PER_CLASS} dla: {emo}",flush=True)
    for t in generate(emo,GEN_PER_CLASS):
        r={c:0 for c in EMOTIONS}; r[emo]=1; r["tekst"]=t; r["aug_source"]=f"llm_{emo}"
        rows.append(r)
aug=pd.DataFrame(rows)[["tekst"]+EMOTIONS+["aug_source"]]
aug=aug.drop_duplicates("tekst").reset_index(drop=True)
aug.to_csv(f"{OUT}/aug_llm_train.csv",index=False)
print("zapisano",len(aug),"przykładów:",{e:int(aug[e].sum()) for e in RARE})
display(aug.groupby("aug_source")["tekst"].head(2))

## Wynik
`/kaggle/working/aug_llm_train.csv` — trafia do `data/processed/`, wejście warunku `5_llm` w 09 i 10.